In [ ]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path

from scipy.stats import gaussian_kde

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))



sys.path.append(f"{project_root}src/")
from plot_utilities import *
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess
import phate


def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config


# ERA5_Graph_Generator.generate_normed_sample() draws the sampled point subset with
# np.random.choice, and nothing in the sampling path re-seeds, so the sampled graph -- and
# every embedding, statistic and figure derived from it -- changes run-to-run unless the
# global RNGs are seeded first. The in-notebook methods pass random_state=42 explicitly and
# the UMAP/concord runner scripts hardcode their own seed, so seeding here is what makes
# the sampled input itself reproducible.
SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()


In [ ]:
# choose data
model_type = 'regular'


# ---------------------------------------------------------------------------
sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator



# Paths
trained_models_path = project_root / 'ERA5' / 'trained_models' / model_type
training_logs_path = trained_models_path / 'logs'
data_path = project_root.parent / 'ERA5' / 'data'  # Adjust if needed

print('Available models:\n')
files = sorted(os.listdir(trained_models_path), reverse=True)

for i, f in enumerate(files):
    if (Path(f).suffix == ".pth") and ('tkernel' in f):
        print(f'[{i}] {f}')


In [ ]:
model_id = 'regular_papermodel2_2026-05-04_t12-01-43'
model, config = load_model(id=model_id , model_type='regular')
config

In [ ]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [ ]:
model = KernelNN(
    width=config['width'],
    ker_width=config['ker_width'],
    depth=config['depth'],
    ker_in=config['edge_attr'],
    in_width=config['node_features'],
    out_width=config['out_width']
).to(device)

# Load trained weights
model_path = f'{trained_models_path}/{model_id}.pth'
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

print(f'Model loaded: {model_id}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

bbox = lon_min, lat_min, lon_max, lat_max
#bbox = -12, 30, 33, 75
Generator = Data_Generator_old(data_path)

N=10000

data = []
data_locations = []

# Seed before the loop: generate_normed_sample() draws the subset with global np.random,
# so without this the sampled points -- and every Lipschitz statistic below -- differ on
# every run. Seeding once here reproduces the whole 31-date sequence exactly.
set_seed()

for date in dates:

    #Generator = Data_Generator(data_path)
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')

    N=5000
    print('generate data')
    graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
    sample = graph.to(device)
    data.append(sample)

In [ ]:
def estimate_lipschitz_constant(batch, output, normalize=True):
    """
    Computes the Lipschitz estimate over graph edges: 
    max_ij ||g(x_i) - g(x_j)|| / ||f(x_i) - f(x_j)||

    Args:
        batch (torch_geometric.data.Data): Graph with attributes:
            - x: tensor (N, 12), original function values f(x)
            - edge_index: tensor (2, E), edge connections
        output (np.ndarray): array (N, 3), embedded function g(x)
        normalize (bool): whether to normalize f and g per feature before computing Lipschitz estimate

    Returns:
        lipschitz_estimate (np.ndarray): array of local Lipschitz ratios for each edge
    """
    device = batch.x.device
    f = batch.x                          # (N, 12)
    g = torch.tensor(output, device=device, dtype=f.dtype)  # (N, 3)
    edge_index = batch.edge_index       # (2, E)

    if normalize:
        # Normalize f per feature
        f_mean = f.mean(dim=0, keepdim=True)
        f_std = f.std(dim=0, unbiased=False, keepdim=True) + 1e-12
        f = (f - f_mean) / f_std

        # Normalize g per feature
        g_mean = g.mean(dim=0, keepdim=True)
        g_std = g.std(dim=0, unbiased=False, keepdim=True) + 1e-12
        g = (g - g_mean) / g_std

    i, j = edge_index[0], edge_index[1]  # indices for edges

    f_diff = f[i] - f[j]                # (E, 12)
    g_diff = g[i] - g[j]                # (E, 3)

    f_norms = torch.norm(f_diff, dim=1) + 1e-12   # (E,)
    g_norms = torch.norm(g_diff, dim=1)           # (E,)

    lipschitz_vals = g_norms / f_norms

    return lipschitz_vals.cpu().numpy()


In [ ]:
normalize = True

nofe_out = []
pca_out = []
tsne_out = []
umap_out = []
phate_out = []
concord_out = []
spatialpca_out = []

#lipschitz_nofe = []
#lipschitz_pca = []
#lipschitz_tsne = []
#lipschitz_umap = []

# -------------------------
# NOFE (model)
# -------------------------
print('NOFE: ', end='')
for i in range(len(data)):
    print(f'{i}/{len(data)}', end='\r')
    batch = data[i].to(device)

    output = model(batch).detach().cpu().numpy()
    nofe_out.append(output)


# -------------------------
# PCA
# -------------------------
print('PCA: ', end='')
pca = PCA(
    n_components=3,
    copy=True,
    whiten=False,
    svd_solver='auto',
    tol=0.0,
    iterated_power='auto',
    n_oversamples=10,
    power_iteration_normalizer='auto',
    random_state=None
)

for i in range(len(data)):
    print(f'{i}/{len(data)}', end='\r')
    batch = data[i]

    X = batch.normed_values.detach().cpu().numpy()
    output = pca.fit_transform(X)
    pca_out.append(output)


# -------------------------
# t-SNE
# -------------------------
print('t-SNE: ', end='')
for i in range(len(data)):
    print(f'{i}/{len(data)}', end='\r')
    batch = data[i]

    X = batch.normed_values.detach().cpu().numpy()

    tsne = TSNE(
        n_components=3,
        perplexity=perplexity,
        learning_rate=200,
        random_state=42
    )

    output = tsne.fit_transform(X)
    tsne_out.append(output)


# -------------------------
# UMAP
# -------------------------
print('UMAP: ', end='')
dir = f'{project_root}/ERA5/umap'

for i in range(len(data)):
    print(f'{i}/{len(data)}', end='\r')
    batch = data[i]

    X = batch.normed_values.detach().cpu().numpy()

    umap_id = f'umap_buffer_{i}'
    np.save(f'{dir}/{umap_id}.npy', X)

    subprocess.run(
        f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}",
        shell=True
    )

    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    umap_out.append(pred_umap)

    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')

# -------------------------
# PHATE
# -------------------------
print('PHATE: ', end='')
for i in range(len(data)):
    print(f'{i}/{len(data)}', end='\r')
    batch = data[i]

    X = batch.normed_values.detach().cpu().numpy()

    phate_op = phate.PHATE(
        n_components=3,
        knn=15,
        mds_solver='smacof',
        random_state=42,
        verbose=False
    )

    output = phate_op.fit_transform(X)
    phate_out.append(output)


# -------------------------
# CONCORD
# -------------------------
print('CONCORD: ', end='')
dir = f'{project_root}/ERA5/concord'

for i in range(len(data)):
    print(f'{i}/{len(data)}', end='\r')
    batch = data[i]

    X = batch.normed_values.detach().cpu().numpy()

    concord_id = f'concord_buffer_{i}'
    np.save(f'{dir}/{concord_id}.npy', X)

    subprocess.run(
        f"cd {dir} && conda run -n concord python execute_concord.py {concord_id}",
        shell=True
    )

    pred_concord = np.load(f'{dir}/res_{concord_id}.npy')
    concord_out.append(pred_concord)

    os.remove(f'{dir}/{concord_id}.npy')
    os.remove(f'{dir}/res_{concord_id}.npy')


# -------------------------
# SpatialPCA
# -------------------------
print('SpatialPCA: ', end='')
dir = f'{project_root}/ERA5/spatialpca'
spatialpca_script = f'{dir}/execute_spatialpca_server.R'
spatialpca_exe = os.environ.get('SPATIALPCA_RSCRIPT', 'Rscript')
# R searches the user library before the conda env's own library; pointing R_LIBS_USER at a
# nonexistent path lets the env's SpatialPCA win (same fix as 2d_scatter.ipynb).
spatialpca_env = dict(os.environ, R_LIBS_USER='/nonexistent')

# Persistent server: one R + SpatialPCA load for all dates instead of one per date.
spatialpca_proc = subprocess.Popen(
    [spatialpca_exe, spatialpca_script],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=dir,
    env=spatialpca_env,
)
if spatialpca_proc.stdout.readline().strip() != 'READY':
    raise RuntimeError('SpatialPCA server did not start')

for i in range(len(data)):
    print(f'{i}/{len(data)}', end='\r')
    batch = data[i]

    X = batch.normed_values.detach().cpu().numpy()
    coords = batch.normed_coords.detach().cpu().numpy()

    spatialpca_id = f'spatialpca_buffer_{i}'
    np.savetxt(f'{dir}/{spatialpca_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{dir}/{spatialpca_id}_coords.csv', coords, delimiter=',')

    spatialpca_proc.stdin.write(f'{spatialpca_id} 3\n')
    spatialpca_proc.stdin.flush()
    reply = spatialpca_proc.stdout.readline().strip()
    if not reply.startswith('OK'):
        raise RuntimeError(f'SpatialPCA failed for {spatialpca_id}: {reply}')

    output = np.loadtxt(f'{dir}/res_{spatialpca_id}.csv', delimiter=',', skiprows=1)
    spatialpca_out.append(output)

    os.remove(f'{dir}/{spatialpca_id}_X.csv')
    os.remove(f'{dir}/{spatialpca_id}_coords.csv')
    os.remove(f'{dir}/res_{spatialpca_id}.csv')

spatialpca_proc.stdin.write('QUIT\n')
spatialpca_proc.stdin.flush()
spatialpca_proc.wait()


In [ ]:
''' # above version should be faster
normalize = True


nofe_out = []
pca_out = []
tsne_out = []
umap_out = []

lipschitz_nofe = []
lipschitz_pca = []
lipschitz_tsne = []
lipschitz_umap = []

for i in range(0, len(data)):
    batch = data[i]
    batch = batch.to(device)
    # NOFE
    output = model(batch.to(device)).detach().cpu().numpy()
    nofe_out.append(output)
    
    # switch to cpu
    X = sample.normed_values.detach().cpu().numpy()
    
    # PCA
    pca = PCA(n_components=3, copy=True, whiten=False, svd_solver='auto', tol=0.0, iterated_power='auto', n_oversamples=10, power_iteration_normalizer='auto', random_state=None)
    output = pca.fit_transform(X)
    pca_out.append(output)

    # TSNE
    tsne = TSNE(n_components=3, perplexity=perplexity, learning_rate=200, random_state=42)
    output = tsne.fit_transform(X)
    tsne_out.append(output)
    
    # UMAP
    umap_id = 'umap_buffer'
    dir = f'{project_root}/ERA5/umap'
    np.save(f'{dir}/{umap_id}.npy', X)
    subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    umap_out.append(pred_umap)
    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')
'''

In [ ]:
normalize=True

lipschitz_nofe = []
lipschitz_pca = []
lipschitz_tsne = []
lipschitz_umap = []
lipschitz_phate = []
lipschitz_concord = []
lipschitz_spatialpca = []

for batch, output in zip(data, nofe_out):
    lip = estimate_lipschitz_constant(batch, output, normalize=normalize)
    lipschitz_nofe.append(lip)
for batch, output in zip(data, pca_out):
    lip = estimate_lipschitz_constant(batch, output, normalize=normalize)
    lipschitz_pca.append(lip)
for batch, output in zip(data, tsne_out):
    lip = estimate_lipschitz_constant(batch, output, normalize=normalize)
    lipschitz_tsne.append(lip)  
for batch, output in zip(data, umap_out):
    lip = estimate_lipschitz_constant(batch, output, normalize=normalize)
    lipschitz_umap.append(lip)  
for batch, output in zip(data, phate_out):
    lip = estimate_lipschitz_constant(batch, output, normalize=normalize)
    lipschitz_phate.append(lip)
for batch, output in zip(data, concord_out):
    lip = estimate_lipschitz_constant(batch, output, normalize=normalize)
    lipschitz_concord.append(lip)
for batch, output in zip(data, spatialpca_out):
    lip = estimate_lipschitz_constant(batch, output, normalize=normalize)
    lipschitz_spatialpca.append(lip)


In [ ]:
lipschitz_nofe_all = np.concatenate(lipschitz_nofe)
lipschitz_pca_all = np.concatenate(lipschitz_pca)
lipschitz_tsne_all = np.concatenate(lipschitz_tsne)
lipschitz_umap_all = np.concatenate(lipschitz_umap)
lipschitz_phate_all = np.concatenate(lipschitz_phate)
lipschitz_concord_all = np.concatenate(lipschitz_concord)
lipschitz_spatialpca_all = np.concatenate(lipschitz_spatialpca)

data = [
    lipschitz_nofe_all,
    lipschitz_pca_all,
    lipschitz_tsne_all,
    lipschitz_umap_all,
    lipschitz_phate_all,
    lipschitz_concord_all,
    lipschitz_spatialpca_all
]

In [ ]:


width = 0.6
ratio = 1.6180339  # Golden ratio : 1.61803398875 

linewidth = 3.31314 *1.1 * 2.54 # 2.54 from inch to cm
plt_width = width * linewidth  # = 5.85 inches
plt_height = plt_width / ratio # Choose a suitable aspect ratio, e.g., 3:2 or 4:3

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

# -------------------------
# Style (paper-ready)
# -------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})


rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(plt_width, plt_height))

positions = np.arange(1, len(data) + 1)

colors = [
    "magenta",  # NOFE
    "blue",     # PCA
    "green",    # t-SNE
    "orange",   # UMAP
    "red",      # PHATE
    "#287C78",  # CONCORD: muted teal
    "#76528B"   # SpatialPCA: muted purple
]

for i, (values, color) in enumerate(zip(data, colors), start=1):
    log_values = np.log10(values)

    # PDF above the row
    if len(values) > 1 and np.std(log_values) > 0:
        grid_log = np.linspace(
            max(log_values.min(), -3),
            min(log_values.max(), 1),
            300
        )
        grid = 10**grid_log

        kde = gaussian_kde(log_values, bw_method=0.1)
        density_log = kde(grid_log)

        density_x = density_log / (grid * np.log(10))
        density_x = density_x / density_x.max() * 0.35

        ax.fill_between(
            grid, i, i + density_x,
            color=color, alpha=0.25, linewidth=0
        )
        ax.plot(grid, i + density_x, color=color, lw=0.9)

    # Scatter observations in the boxplot's row
    N_plot_samples = 500
    sample = (
        values if len(values) <= N_plot_samples
        else rng.choice(values, N_plot_samples, replace=False)
    )

    box_y = i - 0.18

    ax.scatter(
        sample,
        box_y + rng.uniform(-0.045, 0.045, len(sample)),
        s=4, color=color, alpha=0.50, linewidths=0,
        zorder=2
    )

    # Unfilled box plot overlaid on the scattered observations
    bp = ax.boxplot(
        [values],
        positions=[box_y],
        vert=False,
        widths=0.22,
        patch_artist=False,
        showfliers=False,
        whis=1.5,
        manage_ticks=False,
        boxprops=dict(color='black', linewidth=1.0),
        medianprops=dict(color='black', linewidth=1.2),
        whiskerprops=dict(color='black', linewidth=0.9),
        capprops=dict(color='black', linewidth=0.9),
        zorder=3
    )

ax.set_yscale("log")
ax.set_xlim(1e-3, 10)
ax.set_xticks([0.01, 0.1, 1, 10])
ax.set_xticklabels(["0.01", "0.1", "1", "10"])
ax.set_yticks(positions)
ax.set_yticklabels(labels)
ax.set_xlabel("Lipschitz ratio")
#ax.set_ylabel("Method")
ax.grid(axis="x", linestyle=":", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=False, right=False)

fig.tight_layout()
# fig.savefig("raincloud_overlay_boxplots.pdf", bbox_inches="tight")
plt.show()


In [ ]:
fig.savefig("figures/lipschitz_distributions/raincloud_N10000_Jan2019.pdf", bbox_inches="tight")


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

data = [
    np.concatenate(lipschitz_nofe),
    np.concatenate(lipschitz_pca),
    np.concatenate(lipschitz_tsne),
    np.concatenate(lipschitz_umap),
    np.concatenate(lipschitz_phate),
    np.concatenate(lipschitz_concord),
    np.concatenate(lipschitz_spatialpca),
]
data = [x[np.isfinite(x) & (x > 0)] for x in data]


width = 0.6
ratio = 1.6180339  # Golden ratio : 1.61803398875 

linewidth = 3.31314 *1.1 * 2.54 # 2.54 from inch to cm
plt_width = width * linewidth  # = 5.85 inches
plt_height = plt_width / ratio # Choose a suitable aspect ratio, e.g., 3:2 or 4:3

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

# -------------------------
# Style (paper-ready)
# -------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})


rng = np.random.default_rng(42)

fig, ax = plt.subplots(figsize=(plt_width, plt_height))

# Estimate densities in log space
y_grid = np.linspace(-2, 1, 400)

for values, label, color in zip(data, labels, colors):
    log_values = np.log10(values)

    if len(values) < 2 or np.std(log_values) == 0:
        continue

    density = gaussian_kde(log_values)(y_grid)
    ax.plot(10**y_grid, density, color=color, lw=1.5, label=label)

#ax.set_yscale("log")
ax.set_ylim(0, 6)
ax.set_xlim(0, 3)
ax.set_xlabel("Probability Density")
ax.set_ylabel("Lipschitz Ratio")
ax.grid(axis="y", linestyle=":", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=False, right=False)
ax.legend(frameon=False, fontsize=8, ncol=2)

fig.tight_layout()
# fig.savefig("kde.pdf", bbox_inches="tight")
plt.show()


In [ ]:
fig.savefig("figures/lipschitz_distributions/pdf_N10000_Jan2019.pdf", bbox_inches="tight")


---
# From here on only some alternative plot versions

In [ ]:
plt.rcParams.update({
    "font.family": "serif",                 # Use serif fonts
    "font.serif": ["Times New Roman"],      # Set to Times New Roman
    "font.size": 11,          # Base font size
    "axes.labelsize": 11,     # Axis labels
    "legend.fontsize": 10,    # Slightly smaller legend
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.loc": 'upper right',
    # Set font size in points
})

width = 1
ratio = 1.61803398875  # Golden ratio

linewidth = 3.31314 *1.1#* 2.54 # inch
width_in = width * linewidth  # = 5.85 inches
height_in = width_in / ratio # Choose a suitable aspect ratio, e.g., 3:2 or 4:3

lw = 0.4  # Line width for histograms
bins = 80

plt.figure(figsize= (width_in, height_in))

# Add label only for the first element in each loop
for idx, lip in enumerate(lipschitz_nofe):
    plt.hist(lip, bins=bins, histtype='step', density=True, color='magenta',
             label='NOFE' if idx == 0 else None,
             linewidth=lw)

for idx, lip in enumerate(lipschitz_pca):
    plt.hist(lip, bins=bins, histtype='step', density=True, color='blue',
             label='PCA' if idx == 0 else None,
             linewidth=lw)

for idx, lip in enumerate(lipschitz_tsne):
    plt.hist(lip, bins=bins, histtype='step', density=True, color='green',
             label='t-SNE' if idx == 0 else None,
             linewidth=lw)

for idx, lip in enumerate(lipschitz_umap):
    plt.hist(lip, bins=bins, histtype='step', density=True, color='orange',
             label='UMAP' if idx == 0 else None,
             linewidth=lw)

for idx, lip in enumerate(lipschitz_phate):
    plt.hist(lip, bins=bins, histtype='step', density=True, color='red',
             label='PHATE' if idx == 0 else None,
             linewidth=lw)

for idx, lip in enumerate(lipschitz_concord):
    plt.hist(lip, bins=bins, histtype='step', density=True, color='cyan',
             label='CONCORD' if idx == 0 else None,
             linewidth=lw)

for idx, lip in enumerate(lipschitz_spatialpca):
    plt.hist(lip, bins=bins, histtype='step', density=True, color='black',
             label='SpatialPCA' if idx == 0 else None,
             linewidth=lw)

plt.yscale('log')
plt.xlim(0, 10)
plt.xlabel('Lipschitz Ratio')
plt.ylabel('Frequency')
plt.legend()

plt.gcf().set_size_inches(width_in, height_in)
plt.tight_layout()



plt.show()

In [ ]:
lipschitz_nofe_all = np.concatenate(lipschitz_nofe)
lipschitz_pca_all = np.concatenate(lipschitz_pca)
lipschitz_tsne_all = np.concatenate(lipschitz_tsne)
lipschitz_umap_all = np.concatenate(lipschitz_umap)
lipschitz_phate_all = np.concatenate(lipschitz_phate)
lipschitz_concord_all = np.concatenate(lipschitz_concord)
lipschitz_spatialpca_all = np.concatenate(lipschitz_spatialpca)

plt.rcParams.update({
    "font.family": "serif",                 # Use serif fonts
    "font.serif": ["Times New Roman"],      # Set to Times New Roman
    "font.size": 11,          # Base font size
    "axes.labelsize": 11,     # Axis labels
    "legend.fontsize": 10,    # Slightly smaller legend
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.loc": 'upper right',
    # Set font size in points
})

width = 1
ratio = 1.61803398875  # Golden ratio

linewidth = 3.31314 *1.1#* 2.54 # inch
width_in = width * linewidth  # = 5.85 inches
height_in = width_in / ratio # Choose a suitable aspect ratio, e.g., 3:2 or 4:3


lw = 0.4  # Line width for histograms
bins = 80


lw = 2.5
bins = 60
histtype='step'

plt.figure(figsize= (width_in, height_in))

# Add label only for the first element in each loop
plt.hist(lipschitz_nofe_all, bins=bins, histtype='step', density=True, color='magenta',
            label='NOFE',
            linewidth=lw)

plt.hist(lipschitz_pca_all, bins=bins, histtype='step', density=True, color='blue',
            label='PCA',
            linewidth=lw)


plt.hist(lipschitz_tsne_all, bins=bins, histtype='step', density=True, color='green',
            label='t-SNE',
            linewidth=lw)


plt.hist(lipschitz_umap_all, bins=bins, histtype='step', density=True, color='orange',
            label='UMAP',
            linewidth=lw)

plt.hist(lipschitz_phate_all, bins=bins, histtype='step', density=True, color='red',
            label='PHATE',
            linewidth=lw)

plt.hist(lipschitz_concord_all, bins=bins, histtype='step', density=True, color='cyan',
            label='CONCORD',
            linewidth=lw)

plt.hist(lipschitz_spatialpca_all, bins=bins, histtype='step', density=True, color='black',
            label='SpatialPCA',
            linewidth=lw)

plt.yscale('log')
#plt.xlim(0, 10)
plt.xlabel('Lipschitz Ratio')
plt.ylabel('Frequency')
plt.legend()

plt.gcf().set_size_inches(width_in, height_in)
plt.tight_layout()



plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Combine the values for each model
models = {
    "NOFE": np.concatenate(lipschitz_nofe),
    "PCA": np.concatenate(lipschitz_pca),
    "t-SNE": np.concatenate(lipschitz_tsne),
    "UMAP": np.concatenate(lipschitz_umap),
    "PHATE": np.concatenate(lipschitz_phate),
    "CONCORD": np.concatenate(lipschitz_concord),
    "SpatialPCA": np.concatenate(lipschitz_spatialpca),
}

# Publication-style settings
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "figure.dpi": 150,
    "savefig.dpi": 300,
})

# Colors
colors = {
    "NOFE": "#CC79A7",
    "PCA": "#0072B2",
    "t-SNE": "#009E73",
    "UMAP": "#E69F00",
    "PHATE": "#D55E00",
    "CONCORD": "#56B4E9",
    "SpatialPCA": "#555555",
}

# Keep only finite values
models = {
    name: values[np.isfinite(values)]
    for name, values in models.items()
}

# Use common bin edges so the distributions are comparable
all_values = np.concatenate([v for v in models.values() if v.size > 0])
bins = np.histogram_bin_edges(all_values, bins=100)

# Figure dimensions
fig, axes = plt.subplots(
    nrows=len(models),
    ncols=1,
    figsize=(3.6, 7.0),
    sharex=True,
    sharey=True,
    gridspec_kw={"hspace": 0.12},
)

for ax, (name, values) in zip(axes, models.items()):
    ax.hist(
        values,
        bins=bins,
        density=True,
        histtype="stepfilled",
        color=colors[name],
        alpha=0.20,
        edgecolor=colors[name],
        linewidth=1.2,
    )

    # Add a crisp outline
    ax.hist(
        values,
        bins=bins,
        density=True,
        histtype="step",
        color=colors[name],
        linewidth=1.3,
    )

    # Model label inside each subplot
    ax.text(
        0.985, 0.82, name,
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=9,
        fontweight="bold",
    )

    ax.set_yscale("log")
    ax.grid(axis="y", linestyle=":", linewidth=0.6, alpha=0.5)
    ax.tick_params(top=True, right=True, direction="in")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    # Avoid issues with zero-density bins on a logarithmic axis
    ax.set_ylim(bottom=1e-5)

fig.supxlabel("Lipschitz Ratio", y=0.015, fontsize=11)
fig.supylabel("Probability Density (log scale)", x=0.015, fontsize=10)

fig.tight_layout(rect=[0.07, 0.04, 1, 1])

# Optional: save publication-quality output
# fig.savefig("lipschitz_histograms.png", bbox_inches="tight", dpi=300)
# fig.savefig("lipschitz_histograms.pdf", bbox_inches="tight")

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# -------------------------
# Concatenate data
# -------------------------
lipschitz_nofe_all = np.concatenate(lipschitz_nofe)
lipschitz_pca_all  = np.concatenate(lipschitz_pca)
lipschitz_tsne_all = np.concatenate(lipschitz_tsne)
lipschitz_umap_all = np.concatenate(lipschitz_umap)
lipschitz_phate_all = np.concatenate(lipschitz_phate)
lipschitz_concord_all = np.concatenate(lipschitz_concord)
lipschitz_spatialpca_all = np.concatenate(lipschitz_spatialpca)

data = [
    lipschitz_nofe_all,
    lipschitz_pca_all,
    lipschitz_tsne_all,
    lipschitz_umap_all,
    lipschitz_phate_all,
    lipschitz_concord_all,
    lipschitz_spatialpca_all
]

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

# -------------------------
# Style (paper-ready)
# -------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "font.size": 11,
    "axes.labelsize": 11,
    "legend.fontsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

# -------------------------
# Figure sizing (golden ratio)
# -------------------------
width = 2
ratio = 1.61803398875

linewidth = 3.31314 * 1.1
width_in = width * linewidth
height_in = width_in / ratio

plt.figure(figsize=(width_in, height_in))

# -------------------------
# Boxplot
# -------------------------
bp = plt.boxplot(
    data,
    labels=labels,
    patch_artist=True,
    showfliers=False,      # cleaner for paper figures
    widths=0.5
)

# -------------------------
# Style boxes
# -------------------------
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.35)
    patch.set_linewidth(1.5)

for element in ['whiskers', 'caps', 'medians']:
    for line in bp[element]:
        line.set_linewidth(1.5)

# -------------------------
# Axes formatting
# -------------------------
plt.yscale('log')
plt.ylim(1e-2, 10)   # adjust if needed for stability
plt.ylabel("Lipschitz Ratio")

plt.tight_layout()
plt.show()

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

data = [
    np.concatenate(lipschitz_nofe),
    np.concatenate(lipschitz_pca),
    np.concatenate(lipschitz_tsne),
    np.concatenate(lipschitz_umap),
    np.concatenate(lipschitz_phate),
    np.concatenate(lipschitz_concord),
    np.concatenate(lipschitz_spatialpca),
]
data = [x[np.isfinite(x) & (x > 0)] for x in data]

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 10,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(7.0, 3.5))

vp = ax.violinplot(
    data, positions=np.arange(1, 8),
    widths=0.8, showmedians=True,
    showextrema=False, bw_method="scott"
)

for body, color in zip(vp["bodies"], colors):
    body.set_facecolor(color)
    body.set_edgecolor(color)
    body.set_alpha(0.35)
    body.set_linewidth(1.1)

vp["cmedians"].set_color("black")
vp["cmedians"].set_linewidth(1.3)

ax.set_xticks(range(1, 8), labels)
ax.set_yscale("log")
ax.set_ylim(1e-3, 10)
ax.set_ylabel("Lipschitz Ratio")
ax.grid(axis="y", linestyle=":", alpha=0.35)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=True, right=True)

fig.tight_layout()
# fig.savefig("violin.pdf", bbox_inches="tight")
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde



width = 1
ratio = 1.61803398875  # Golden ratio

linewidth = 3.31314 *1.1 * 2.54#* 2.54 # inch
width_in = width * linewidth  # = 5.85 inches
height_in = width_in / ratio # Choose a suitable aspect ratio, e.g., 3:2 or 4:3


labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

data = [
    np.concatenate(lipschitz_nofe),
    np.concatenate(lipschitz_pca),
    np.concatenate(lipschitz_tsne),
    np.concatenate(lipschitz_umap),
    np.concatenate(lipschitz_phate),
    np.concatenate(lipschitz_concord),
    np.concatenate(lipschitz_spatialpca),
]
data = [x[np.isfinite(x) & (x > 0)] for x in data]

rng = np.random.default_rng(42)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(width_in, height_in))

# Estimate density in log space for a meaningful log-axis shape
for i, (values, color) in enumerate(zip(data, colors), start=1):
    log_values = np.log10(values)
    grid = np.linspace(
        max(log_values.min(), -2),
        min(log_values.max(), 1),
        250
    )

    if len(values) > 1 and np.std(log_values) > 0:
        kde = gaussian_kde(log_values)
        density = kde(grid)
        density = density / density.max() * 0.35

        # Half violin, to the left of each category position
        ax.fill_betweenx(
            10**grid, i - density, i,
            color=color, alpha=0.3, linewidth=0
        )
        ax.plot(i - density, 10**grid, color=color, lw=0.9)

    # Boxplot to the right
    bp = ax.boxplot(
        [values], positions=[i + 0.13],
        widths=0.16, vert=True, patch_artist=True,
        showfliers=False
    )
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.35)
    for key in ["whiskers", "caps", "medians"]:
        for line in bp[key]:
            line.set_color("black")
            line.set_linewidth(0.8)

    # Sample at most 250 observations to avoid overplotting
    sample = values if len(values) <= 250 else rng.choice(values, 250, replace=False)
    ax.scatter(
        i + 0.13 + rng.uniform(0.10, 0.25, len(sample)),
        sample,
        s=4, color=color, alpha=0.35, linewidths=0
    )

ax.set_xticks(range(1, 8), labels)
ax.set_yscale("log")
ax.set_ylim(1e-3, 10)
ax.set_ylabel("Lipschitz Ratio")
ax.grid(axis="y", linestyle=":", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=True, right=True)

fig.tight_layout()
# fig.savefig("raincloud.pdf", bbox_inches="tight")
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

data = [
    np.concatenate(lipschitz_nofe),
    np.concatenate(lipschitz_pca),
    np.concatenate(lipschitz_tsne),
    np.concatenate(lipschitz_umap),
    np.concatenate(lipschitz_phate),
    np.concatenate(lipschitz_concord),
    np.concatenate(lipschitz_spatialpca),
]
data = [x[np.isfinite(x) & (x > 0)] for x in data]

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(7.0, 3.8))

y_grid = np.linspace(-2, 1, 350)  # log10 Lipschitz Ratio

for i, (values, color, label) in enumerate(zip(data, colors, labels)):
    log_values = np.log10(values)

    if len(values) < 2 or np.std(log_values) == 0:
        continue

    density = gaussian_kde(log_values)(y_grid)
    density = density / density.max() * 0.38

    # Ridge extends horizontally; value is vertical
    ax.fill_betweenx(
        10**y_grid, i - density, i + density,
        color=color, alpha=0.25, linewidth=0
    )
    ax.plot(i + density, 10**y_grid, color=color, lw=1.0)
    ax.plot(i - density, 10**y_grid, color=color, lw=1.0)

ax.set_xticks(range(7), labels)
ax.set_yscale("log")
ax.set_ylim(1e-2, 10)
ax.set_ylabel("Lipschitz Ratio")
ax.set_xlim(-0.6, 6.6)
ax.grid(axis="y", linestyle=":", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=True, right=True)

fig.tight_layout()
# fig.savefig("ridgeline.pdf", bbox_inches="tight")
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

data = [
    np.concatenate(lipschitz_nofe),
    np.concatenate(lipschitz_pca),
    np.concatenate(lipschitz_tsne),
    np.concatenate(lipschitz_umap),
    np.concatenate(lipschitz_phate),
    np.concatenate(lipschitz_concord),
    np.concatenate(lipschitz_spatialpca),
]
data = [x[np.isfinite(x) & (x > 0)] for x in data]

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(6.8, 3.6))

# Estimate densities in log space
y_grid = np.linspace(-2, 1, 400)

for values, label, color in zip(data, labels, colors):
    log_values = np.log10(values)

    if len(values) < 2 or np.std(log_values) == 0:
        continue

    density = gaussian_kde(log_values)(y_grid)
    ax.plot(density, 10**y_grid, color=color, lw=1.5, label=label)

ax.set_yscale("log")
ax.set_ylim(1e-2, 10)
ax.set_xlabel("Probability Density (per log10 unit)")
ax.set_ylabel("Lipschitz Ratio")
ax.grid(axis="y", linestyle=":", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(frameon=False, fontsize=8, ncol=2)

fig.tight_layout()
# fig.savefig("kde.pdf", bbox_inches="tight")
plt.show()


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

data = [
    np.concatenate(lipschitz_nofe),
    np.concatenate(lipschitz_pca),
    np.concatenate(lipschitz_tsne),
    np.concatenate(lipschitz_umap),
    np.concatenate(lipschitz_phate),
    np.concatenate(lipschitz_concord),
    np.concatenate(lipschitz_spatialpca),
]
data = [x[np.isfinite(x) & (x > 0)] for x in data]

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(6.8, 3.6))

# Estimate densities in log space
y_grid = np.linspace(-2, 1, 400)

for values, label, color in zip(data, labels, colors):
    log_values = np.log10(values)

    if len(values) < 2 or np.std(log_values) == 0:
        continue

    density = gaussian_kde(log_values)(y_grid)
    ax.plot(10**y_grid, density, color=color, lw=1.5, label=label)

#ax.set_yscale("log")
ax.set_ylim(0, 6)
ax.set_xlim(0, 3)
ax.set_xlabel("Probability Density (per log10 unit)")
ax.set_ylabel("Lipschitz Ratio")
ax.grid(axis="y", linestyle=":", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=False, right=False)
ax.legend(frameon=False, fontsize=8, ncol=2)

fig.tight_layout()
# fig.savefig("kde.pdf", bbox_inches="tight")
plt.show()


In [ ]:
len(np.concatenate(lipschitz_nofe))

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

data = [
    lipschitz_nofe_all,
    lipschitz_pca_all,
    lipschitz_tsne_all,
    lipschitz_umap_all,
    lipschitz_phate_all,
    lipschitz_concord_all,
    lipschitz_spatialpca_all
]
data = [x[np.isfinite(x) & (x > 0)] for x in data]

rng = np.random.default_rng(42)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(7.0, 3.8))

positions = np.arange(1, len(data) + 1)

for i, (values, color) in enumerate(zip(data, colors), start=1):
    log_values = np.log10(values)

    # Density estimated in log space, displayed on original ratio axis
    if len(values) > 1 and np.std(log_values) > 0:
        grid_log = np.linspace(
            max(log_values.min(), -3),
            min(log_values.max(), 1),
            300
        )
        grid = 10**grid_log
        kde = gaussian_kde(log_values, bw_method=0.2)
        density_log = kde(grid_log)

        # Convert density per log10 unit to density per original x unit
        density_x = density_log / (grid * np.log(10))
        density_x = density_x / density_x.max() * 0.35

        ax.fill_between(
            grid, i, i + density_x,
            color=color, alpha=0.25, linewidth=0
        )
        ax.plot(grid, i + density_x, color=color, lw=0.9)

    # Boxplot overlay
    '''
    bp = ax.boxplot(
        [values], positions=[i], widths=0.18,
        patch_artist=True, showfliers=False
    )
    bp["boxes"][0].set_facecolor(color)
    bp["boxes"][0].set_alpha(0.45)

    for key in ["whiskers", "caps", "medians"]:
        for line in bp[key]:
            line.set_color("black")
            line.set_linewidth(0.8)
    '''

    # Jitter a sample of observations
    N_plot_samples = 500
    sample = values if len(values) <= N_plot_samples else rng.choice(values, N_plot_samples, replace=False)
    ax.scatter(
        sample,
        i - rng.uniform(0.08, 0.22, len(sample)),
        s=5, color=color, alpha=0.35, linewidths=0
    )

ax.set_xscale("log")
ax.set_xlim(1e-3, 10)
ax.set_xticks([0.01, 0.1, 1, 10])
ax.set_xticklabels(["0.01", "0.1", "1", "10"])
ax.set_yticks(positions)
ax.set_yticklabels(labels)
ax.set_xlabel("Lipschitz Ratio")
ax.set_ylabel("Method")
ax.grid(axis="x", linestyle=":", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=True, right=True)

fig.tight_layout()
# fig.savefig("raincloud_original_axes.pdf", bbox_inches="tight")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

labels = ["NOFE", "PCA", "t-SNE", "UMAP", "PHATE", "CONCORD", "SpatialPCA"]
colors = ["magenta", "blue", "green", "orange", "red", "cyan", "black"]

data = [
    np.concatenate(lipschitz_nofe),
    np.concatenate(lipschitz_pca),
    np.concatenate(lipschitz_tsne),
    np.concatenate(lipschitz_umap),
    np.concatenate(lipschitz_phate),
    np.concatenate(lipschitz_concord),
    np.concatenate(lipschitz_spatialpca),
]
data = [x[np.isfinite(x) & (x > 0)] for x in data]

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(6.8, 3.6))

# Evaluate the KDE on a logarithmically spaced grid
x_grid = np.geomspace(1e-2, 10, 500)

for values, label, color in zip(data, labels, colors):
    if len(values) < 2 or np.std(values) == 0:
        continue

    # KDE in log space, then transform to density per original x unit
    log_values = np.log10(values)
    kde = gaussian_kde(log_values)
    density_log = kde(np.log10(x_grid))
    density_x = density_log / (x_grid * np.log(10))

    ax.plot(x_grid, density_x, color=color, lw=1.5, label=label)

ax.set_xscale("log")
ax.set_xlim(1e-2, 10)
ax.set_xlabel("Lipschitz Ratio")
ax.set_ylabel("Probability Density")
ax.set_xticks([0.01, 0.1, 1, 10])
ax.set_xticklabels(["0.01", "0.1", "1", "10"])
ax.grid(axis="x", linestyle=":", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(direction="in", top=True, right=True)
ax.legend(frameon=False, fontsize=8, ncol=2)

fig.tight_layout()
# fig.savefig("kde_original_axes.pdf", bbox_inches="tight")
plt.show()


In [ ]:
def mean1(data:list):
    means = []
    for d in data:
        means.append(np.mean(d))
    return np.mean(means)

def mean2(data:list):
    full_data = data[0]
    for d in range(1,len(data)):
        full_data = np.concatenate((full_data, data[d]), axis=0)
    return np.mean(full_data)

def var1(data:list):
    vars = []
    for d in data:
        vars.append(np.var(d))
    return np.mean(vars)

def var2(data:list):
    full_data = data[0]
    for d in range(1,len(data)):
        full_data = np.concatenate((full_data, data[d]), axis=0)
    return np.var(full_data)

In [ ]:
print('Mean method 1 (mean of means):')
print(f'NOFE: {mean1(lipschitz_nofe)}')
print(f'PCA: {mean1(lipschitz_pca)}')
print(f't-SNE: {mean1(lipschitz_tsne)}')
print(f'UMAP: {mean1(lipschitz_umap)}') 
print(f'PHATE: {mean1(lipschitz_phate)}')
print(f'CONCORD: {mean1(lipschitz_concord)}')
print(f'SpatialPCA: {mean1(lipschitz_spatialpca)}')

print('\nMean method 2 (mean of full data):')
print(f'NOFE: {mean2(lipschitz_nofe)}')
print(f'PCA: {mean2(lipschitz_pca)}')
print(f't-SNE: {mean2(lipschitz_tsne)}')
print(f'UMAP: {mean2(lipschitz_umap)}')
print(f'PHATE: {mean2(lipschitz_phate)}')
print(f'CONCORD: {mean2(lipschitz_concord)}')
print(f'SpatialPCA: {mean2(lipschitz_spatialpca)}')

print('Variance method 1 (mean of variances):')
print(f'NOFE: {var1(lipschitz_nofe)}')
print(f'PCA: {var1(lipschitz_pca)}')
print(f't-SNE: {var1(lipschitz_tsne)}')
print(f'UMAP: {var1(lipschitz_umap)}')
print(f'PHATE: {var1(lipschitz_phate)}')
print(f'CONCORD: {var1(lipschitz_concord)}')
print(f'SpatialPCA: {var1(lipschitz_spatialpca)}')

print('\nVariance method 2 (variance of full data):')
print(f'NOFE: {var2(lipschitz_nofe)}')
print(f'PCA: {var2(lipschitz_pca)}')
print(f't-SNE: {var2(lipschitz_tsne)}')
print(f'UMAP: {var2(lipschitz_umap)}')
print(f'PHATE: {var2(lipschitz_phate)}')
print(f'CONCORD: {var2(lipschitz_concord)}')
print(f'SpatialPCA: {var2(lipschitz_spatialpca)}')

In [ ]:
np.median(lipschitz_nofe_all), np.median(lipschitz_pca_all), np.median(lipschitz_tsne_all), np.median(lipschitz_umap_all)